In [1]:
!pip install sympy



In [2]:
!pip install kanren

In [3]:
!pip install minikanren

In [4]:
import json 
from kanren import run, facts, eq, Relation, var, conde


In [5]:
def parent(x, y):  
    return conde ([father(x, y)], [mother(x, y)])  

def grandparent(x, y):  
    temp = var()  
    return conde ((parent(x, temp), parent(temp, y)))  

def sibling(x, y):  
    temp = var()  
    return conde ((parent(x, temp), parent(temp, y)))  

def uncle(x, y):  
    temp = var()  
    return conde((father(temp, x), sibling(temp, y)))


In [6]:
if __name__ == '__main__':  
    father = Relation()  
    mother = Relation()  

    with open('relationships.json') as f:  
        d = json.loads(f.read())  

    for item in d['father']:  
        facts(father, (list(item.keys())[0], list(item.values())[0]))  

    for item in d['mother']:  
        facts(mother, (list(item.keys())[0], list(item.values())[0]))  

    x = var()

In [7]:
name = 'John'  
output = run(0, x, father(name, x))  

print(name + ' children are: ')  
for item in output:  
    print(item)

John children are: 
Adam
David
William


In [8]:
name = 'Adam'  
output = run(0, x, father(name, x))  

print(name + ' children are: ')  
for item in output:  
    print(item)

Adam children are: 
Sophia


In [9]:
name = 'Megan'  
output = run(0, x, grandparent(name, x))  

print(name + ' grandparent are: ')  
for item in output:  
    print(item)

Megan grandparent are: 
Peter
Stephanie
Sophia
Neil
Chris
Julie
Tiffany
Wayne


In [10]:
name = 'Megan'  
output = run(0, x, sibling(name, x))  

print(name + ' Siblings are: ')  
for item in output:  
    print(item)

Megan Siblings are: 
Peter
Stephanie
Sophia
Neil
Chris
Julie
Tiffany
Wayne


In [11]:
name = 'Tiffany'  
output = run(0, x, uncle(name, x))  

print(name + ' uncles are: ')  
for item in output:  
    print(item)

Tiffany uncles are: 


# Script 1 to find David Siblings, Tiffany’s uncles,  all spouses


In [12]:
import json
from kanren import run, facts, eq, Relation, var, conde

# Define parent relationship
def parent(x, y):  
    return conde([father(x, y)], [mother(x, y)])  

# Define grandparent relationship
def grandparent(x, y):  
    temp = var()  
    return conde((parent(x, temp), parent(temp, y)))  

# Define sibling relationship (excluding self)
def sibling(x, y):  
    temp = var()  
    return conde((parent(temp, x), parent(temp, y)))  

# Define uncle relationship (father’s or mother’s brother)
def uncle(x, y):  
    parent_var = var()
    return conde(
        (father(parent_var, y), sibling(x, parent_var)),  # Father's brother
        (mother(parent_var, y), sibling(x, parent_var))   # Mother's brother
    )

# Define spouse relationship
def spouse(x, y):  
    return conde([married(x, y)], [married(y, x)])  # Ensure bidirectional lookup

# Main execution block
if __name__ == '__main__':  
    father = Relation()  
    mother = Relation()  
    married = Relation()  # New spouse relation

    # Load relationships from JSON file
    with open('relationships.json') as f:  
        d = json.loads(f.read())  

    # Add father facts
    for item in d.get('father', []):  
        facts(father, (list(item.keys())[0], list(item.values())[0]))  

    # Add mother facts
    for item in d.get('mother', []):  
        facts(mother, (list(item.keys())[0], list(item.values())[0]))  

    # Add spouse facts
    for item in d.get('spouse', []):  
        facts(married, (list(item.keys())[0], list(item.values())[0]))  
        facts(married, (list(item.values())[0], list(item.keys())[0]))  # Ensure both directions

    x = var()

    # 🔹 Query 1: Find David’s siblings (filter out self-match + remove duplicates)
    name = 'David'  
    siblings = run(0, x, sibling(name, x))  
    siblings = sorted(set(s for s in siblings if s != name))  # Remove self-match + duplicates

    print(f"{name}'s siblings are: {', '.join(siblings) if siblings else 'None'}")

    # 🔹 Query 2: Find Tiffany’s uncles (remove duplicates)
    name = 'Tiffany'  
    uncles = sorted(set(run(0, x, uncle(x, name))))  

    print(f"{name}'s uncles are: {', '.join(uncles) if uncles else 'None'}")

    # 🔹 Query 3: Find all spouses (remove duplicates)
    spouse_list = sorted(set(run(0, x, spouse(x, var()))))  

    print("Spouses:", ', '.join(spouse_list) if spouse_list else "None")


David's siblings are: Adam, William
Tiffany's uncles are: Adam, David, William
Spouses: Adam, David, Emma, John, Lily, Megan, Olivia, William


# Script 2 to find David Siblings, Tiffany’s uncles, all spouses

In [13]:
import json
from kanren import run, facts, eq, Relation, var, conde

# Define relationships
father = Relation()
mother = Relation()
married = Relation()

# Parent relationship
def parent(x, y):
    return conde([father(x, y)], [mother(x, y)])

# Sibling relationship (ensuring x ≠ y)
def sibling(x, y):
    temp = var()
    return conde(
        (parent(temp, x), parent(temp, y), neq(x, y))  # Ensure common parent but x ≠ y
    )

# Grandparent relationship
def grandparent(x, y):
    temp = var()
    return conde((parent(x, temp), parent(temp, y)))

# Uncle relationship (father's or mother's brother)
def uncle(x, y):
    parent_var = var()
    return conde(
        (parent(parent_var, y), sibling(x, parent_var))  # x is sibling of y’s parent
    )

# Spouse relationship (ensure bidirectional lookup)
def spouse(x, y):
    return conde([married(x, y)], [married(y, x)])

# Helper function to check inequality (neq)
def neq(x, y):
    return conde([eq(x, y)], [])

# Load relationships from JSON file
with open('relationships.json') as f:
    d = json.loads(f.read())

# Add father facts
for item in d.get('father', []):
    facts(father, (list(item.keys())[0], list(item.values())[0]))

# Add mother facts
for item in d.get('mother', []):
    facts(mother, (list(item.keys())[0], list(item.values())[0]))

# Add spouse facts (ensure bidirectionality)
for item in d.get('spouse', []):
    facts(married, (list(item.keys())[0], list(item.values())[0]))
    facts(married, (list(item.values())[0], list(item.keys())[0]))

x = var()

# Query 1: Find David’s siblings (remove self-match + duplicates)
name = 'David'
siblings = run(0, x, sibling(name, x))
siblings = sorted(set(s for s in siblings if s != name))  # Remove self-match + duplicates
print(f"{name}'s siblings are: {', '.join(siblings) if siblings else 'None'}")

# Query 2: Find Tiffany’s uncles (remove duplicates)
name = 'Tiffany'
uncles = sorted(set(run(0, x, uncle(x, name))))
print(f"{name}'s uncles are: {', '.join(uncles) if uncles else 'None'}")

# Query 3: Find all spouses (remove duplicates)
spouse_list = sorted(set(run(0, x, spouse(x, var()))))
print("Spouses:", ', '.join(spouse_list) if spouse_list else "None")


David's siblings are: Adam, William
Tiffany's uncles are: Adam, David, William
Spouses: Adam, David, Emma, John, Lily, Megan, Olivia, William


# Script 3 to find David Siblings, Tiffany’s uncles, all spouses


In [14]:
import json
from kanren import run, facts, eq, Relation, var, conde

# Define relationships
father = Relation()
mother = Relation()
married = Relation()

# Parent relationship
def parent(x, y):
    return conde([father(x, y)], [mother(x, y)])

# Sibling relationship (ensuring x ≠ y)
def sibling(x, y):
    temp = var()
    return conde(
        (parent(temp, x), parent(temp, y), neq(x, y))  # Common parent, but x ≠ y
    )

# Uncle/Aunt relationship (Fixing the incorrect result)
def uncle_or_aunt(x, y):
    parent_var = var()
    return conde(
        (parent(parent_var, y), sibling(x, parent_var), neq(x, parent_var))  # Remove self-match!
    )

# Spouse relationship (ensure bidirectional lookup)
def spouse(x, y):
    return conde([married(x, y)], [married(y, x)])

# Helper function for inequality check
def neq(x, y):
    return conde([eq(x, y)], [])

# New and improved JSON data
data = {
    "father": [
        {"John": "William"},
        {"John": "David"},
        {"John": "Adam"},
        {"William": "Chris"},
        {"William": "Stephanie"},
        {"David": "Tiffany"},
        {"David": "Neil"},
        {"Adam": "Sophia"},
        {"Adam": "Liam"}
    ],
    "mother": [
        {"Megan": "William"},
        {"Megan": "David"},
        {"Megan": "Adam"},
        {"Emma": "Stephanie"},
        {"Emma": "Chris"},
        {"Olivia": "Tiffany"},
        {"Olivia": "Neil"},
        {"Lily": "Sophia"},
        {"Lily": "Liam"}
    ],
    "spouse": [
        {"John": "Megan"},
        {"William": "Emma"},
        {"David": "Olivia"},
        {"Adam": "Lily"}
    ]
}

# Add facts for relationships
for relation, rel_data in [("father", father), ("mother", mother)]:
    for item in data[relation]:
        facts(rel_data, (list(item.keys())[0], list(item.values())[0]))

# Add spouse facts (ensuring bidirectionality)
for item in data["spouse"]:
    facts(married, (list(item.keys())[0], list(item.values())[0]))
    facts(married, (list(item.values())[0], list(item.keys())[0]))

x = var()

# ✅ Query 1: Find David’s siblings (Remove self-match!)
name = 'David'
siblings = sorted(set(run(0, x, sibling(name, x))) - {name})  # Remove self
print(f"{name}'s siblings are: {', '.join(siblings) if siblings else 'None'}")

# ✅ Query 2: Find Tiffany’s uncles/aunts (Now fixed!)
name = 'Tiffany'
uncles_aunts = sorted(set(run(0, x, uncle_or_aunt(x, name))))
print(f"{name}'s uncles/aunts are: {', '.join(uncles_aunts) if uncles_aunts else 'None'}")

# ✅ Query 3: Find all spouses (Checking bidirectionality)
spouse_list = sorted(set(run(0, x, spouse(x, var()))))
print("Spouses:", ', '.join(spouse_list) if spouse_list else "None")


David's siblings are: Adam, William
Tiffany's uncles/aunts are: Adam, David, William
Spouses: Adam, David, Emma, John, Lily, Megan, Olivia, William
